# ChromaDB Explorer

Interactive notebook for exploring and testing your vector database.

In [1]:
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd

# Connect to database
client = chromadb.PersistentClient(path="../chroma_data")
print("Connected to ChromaDB")

Connected to ChromaDB


## 1. View All Collections

In [2]:
# List all collections
collections = client.list_collections()

print(f"Found {len(collections)} collections:\n")
for coll in collections:
    print(f"- {coll.name}: {coll.count()} documents")
    print(f"  Metadata: {coll.metadata}\n")

Found 1 collections:

- sample_docs: 8 documents
  Metadata: {'description': 'Sample documents for demo'}



## 2. Explore a Collection

In [3]:
# Get collection (change name if needed)
collection = client.get_collection("sample_docs")

# Get all documents
results = collection.get()

# Create DataFrame for easy viewing
df = pd.DataFrame({
    'ID': results['ids'],
    'Text': results['documents'],
    'Metadata': results['metadatas']
})

print(f"Collection: {collection.name}")
print(f"Total documents: {len(df)}\n")
df

Collection: sample_docs
Total documents: 8



,ID,Text,Metadata
0,python-1,Python is a high-level programming language kn...,"{'topic': 'programming', 'language': 'python',..."
1,javascript-1,JavaScript is the programming language of the ...,"{'difficulty': 'beginner', 'topic': 'programmi..."
2,ml-1,Machine learning is a subset of artificial int...,"{'topic': 'ai', 'difficulty': 'intermediate', ..."
3,nn-1,Neural networks are computing systems inspired...,"{'subtopic': 'neural-networks', 'topic': 'ai',..."
4,rag-1,RAG (Retrieval Augmented Generation) combines ...,"{'topic': 'ai', 'subtopic': 'rag', 'difficulty..."
5,vector-db-1,Vector databases store embeddings and enable s...,"{'topic': 'databases', 'difficulty': 'intermed..."
6,web-scraping-1,Web scraping extracts data from websites using...,"{'topic': 'data-collection', 'subtopic': 'scra..."
7,embeddings-1,Embeddings are vector representations of text ...,"{'topic': 'ai', 'difficulty': 'intermediate', ..."


## 3. Semantic Search

In [4]:
# Search for similar documents
query = "How do I build AI applications?"

results = collection.query(
    query_texts=[query],
    n_results=5
)

print(f"Query: '{query}'\n")
print("Top 5 Results:\n" + "="*80)

for i, (doc, distance, metadata) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
), 1):
    print(f"\n{i}. Distance: {distance:.3f}")
    print(f"   Metadata: {metadata}")
    print(f"   Text: {doc[:150]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: 'How do I build AI applications?'

Top 5 Results:

1. Distance: 0.588
   Metadata: {'topic': 'ai', 'subtopic': 'machine-learning', 'difficulty': 'intermediate'}
   Text: Machine learning is a subset of artificial intelligence that enables computers to learn from data. Popular frameworks include TensorFlow and PyTorch....

2. Distance: 0.705
   Metadata: {'difficulty': 'advanced', 'subtopic': 'neural-networks', 'topic': 'ai'}
   Text: Neural networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes that process information....

3. Distance: 0.734
   Metadata: {'topic': 'databases', 'subtopic': 'vector-db', 'difficulty': 'intermediate'}
   Text: Vector databases store embeddings and enable semantic search. Popular options include ChromaDB, Pinecone, and Weaviate....

4. Distance: 0.751
   Metadata: {'difficulty': 'beginner', 'topic': 'programming', 'language': 'python'}
   Text: Python is a high-level programming language

## 4. Search with Metadata Filter

In [5]:
# Search only AI-related documents
query = "machine learning"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"topic": "ai"}  # Filter by metadata
)

print(f"Query: '{query}' (filtered by topic='ai')\n")

for i, (doc, metadata) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0]
), 1):
    print(f"\n{i}. {metadata}")
    print(f"   {doc[:100]}...")

Query: 'machine learning' (filtered by topic='ai')


1. {'difficulty': 'intermediate', 'subtopic': 'machine-learning', 'topic': 'ai'}
   Machine learning is a subset of artificial intelligence that enables computers to learn from data. P...

2. {'topic': 'ai', 'difficulty': 'advanced', 'subtopic': 'neural-networks'}
   Neural networks are computing systems inspired by biological neural networks. They consist of layers...

3. {'topic': 'ai', 'difficulty': 'advanced', 'subtopic': 'rag'}
   RAG (Retrieval Augmented Generation) combines information retrieval with language model generation. ...


## 5. Try Your Own Searches

Modify and run this cell with your own queries:

In [6]:
# Your custom search
my_query = "neural networks"  # Change this

results = collection.query(
    query_texts=[my_query],
    n_results=3
)

# Display as DataFrame
search_df = pd.DataFrame({
    'Distance': results['distances'][0],
    'Metadata': results['metadatas'][0],
    'Text': [doc[:100] + '...' for doc in results['documents'][0]]
})

print(f"Search: '{my_query}'\n")
search_df

Search: 'neural networks'



,Distance,Metadata,Text
0,0.401222,"{'topic': 'ai', 'difficulty': 'advanced', 'sub...",Neural networks are computing systems inspired...
1,0.552711,"{'difficulty': 'intermediate', 'subtopic': 'ma...",Machine learning is a subset of artificial int...
2,0.798649,"{'difficulty': 'intermediate', 'subtopic': 'em...",Embeddings are vector representations of text ...


## 6. Add New Documents (Optional)

In [7]:
# Uncomment to add new documents

# collection.add(
#     documents=["Your new document text here"],
#     ids=["new-doc-1"],
#     metadatas=[{"topic": "custom", "source": "manual"}]
# )
# 
# print(f"Added document. Total: {collection.count()}")

## Tips

- Run cells with Shift+Enter
- Modify queries and re-run cells to experiment
- Use pandas DataFrames for clean visualization
- Lower distance = more similar/relevant
- Try different metadata filters